# Sim 2b + 3 auf Gemma-4: PDA mit PLE und Shared KV-Cache

Vergleich mit Qwen3-Ergebnissen. Gemma-4 hat architektonische Features die fuer PDA relevant sind:
- **Per-Layer Embeddings (PLE)**: Separater Conditioning-Kanal pro Layer
- **Shared KV-Cache**: Spaete Layer teilen K/V — beweist Redundanz
- **Alternating Local/Global Attention**: Unterschiedliche Informationsfluss-Muster

Hypothese: Gemma-4 hat *reichere* Subspace-Struktur als Qwen3, weil PLE die
Informationsverteilung ueber Layer verändert.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import gc
import sys, os

sys.path.append(".")
import sim_gemma4_helpers as helpers

# === CONFIG ===
MODEL_ID = "google/gemma-4-4b-it"  # E4B: ~3.65 GB in 4-bit
QUANTIZE = True  # 4-bit fuer 12GB GPU

print(f"Loading {MODEL_ID} (4-bit={QUANTIZE})...")
model, tokenizer = helpers.load_gemma4(MODEL_ID, quantize_4bit=QUANTIZE)

info = helpers.get_model_info(model)
print(f"\nModel Info:")
for k, v in info.items():
    print(f"  {k}: {v}")

n_layers = info['num_layers']
d_model = info['d_model']
print(f"\nVRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Prompts (identisch zu Sim 2b fuer Vergleichbarkeit)
prompts_facts = [
    "The capital of France is Paris.",
    "The chemical symbol for water is H2O.",
    "Einstein is known for the theory of relativity.",
    "The sun is a star in the center of the solar system.",
    "Humans breathe oxygen to survive."
]
prompts_reasoning = [
    "If A > B and B > C, then A must be greater than C.",
    "To solve x + 5 = 10, we subtract 5 from both sides.",
    "The next number in the sequence 2, 4, 8, 16 is 32.",
    "A triangle with three equal sides is called equilateral.",
    "If it rains, the ground gets wet. It is raining, so the ground is wet."
]
prompts_creative = [
    "Once upon a time in a galaxy far, far away,",
    "The neon lights of the city reflected in the puddles,",
    "A giant clockwork dragon roared over the mountain peak,",
    "The secret of the universe was hidden in a small tea cup,",
    "Music filled the air as the stars began to dance."
]
all_prompts = prompts_facts + prompts_reasoning + prompts_creative
print(f"\n{len(all_prompts)} Prompts geladen.")

## Sim 2b: Dominant Direction Analysis

Kernfrage: Ist die dominante Richtung bei Gemma-4 genauso extrem wie bei Qwen3?

In [ ]:
# Layer-Auswahl: frueh, 1/4, 1/2, 3/4, spaet
layers_to_check = sorted(set([
    1,
    n_layers // 4,
    n_layers // 2,
    3 * n_layers // 4,
    n_layers - 2
]))

# KV-Shared-Grenze markieren
kv_shared_start = info.get('kv_shared_start', n_layers)
print(f"Layer: {layers_to_check}")
print(f"KV-Shared ab Layer: {kv_shared_start}")
print(f"PLE aktiv: {info.get('has_ple', False)}")

# Aktivierungen extrahieren
acts = helpers.extract_activations(model, tokenizer, all_prompts, layers_to_check)

# Dominant Direction Analyse
results = {}
for l in layers_to_check:
    shared = " [KV-SHARED]" if l >= kv_shared_start else ""
    results[l] = helpers.analyze_dominant_direction(acts[l], f"Gemma-4 Layer {l}{shared}")

In [ ]:
# Visualisierung: Vergleich mit Qwen3-Ergebnissen
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

layers = sorted(results.keys())
positions = [f"L{l}" for l in layers]

# PR Original vs Residual
pr_orig = [results[l]['pr_original'] for l in layers]
pr_resid = [results[l]['pr_residual'] for l in layers]
x = np.arange(len(layers))
w = 0.35
axes[0].bar(x - w/2, pr_orig, w, label='Original', color='steelblue')
axes[0].bar(x + w/2, pr_resid, w, label='After removal', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(positions)
axes[0].set_title('Participation Ratio: Original vs Mean-Removed')
axes[0].legend()
axes[0].grid(True, axis='y')

# Dominant Variance Share
dom_var = [results[l]['dominant_var_share'] for l in layers]
colors = ['red' if l >= kv_shared_start else 'steelblue' for l in layers]
axes[1].bar(x, dom_var, color=colors)
axes[1].set_xticks(x)
axes[1].set_xticklabels(positions)
axes[1].set_title('Dominant Direction Variance Share')
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, axis='y')
axes[1].legend(['KV-shared' if l >= kv_shared_start else 'Normal' for l in layers[:2]])

# Cumulative Variance (residual) per layer
for l in layers:
    cv = results[l]['cum_var_residual'][:100]
    style = '--' if l >= kv_shared_start else '-'
    axes[2].plot(cv, style, label=f"L{l}")
axes[2].axhline(y=0.9, color='gray', linestyle=':', alpha=0.5)
axes[2].set_title('Cumulative Variance (residual) per Layer')
axes[2].set_xlabel('Component')
axes[2].legend(fontsize=8)
axes[2].grid(True)

plt.suptitle(f'Gemma-4 Sim 2b: Dominant Direction Analysis ({MODEL_ID})', fontsize=14)
plt.tight_layout()
plt.show()

# Vergleichstabelle
print(f"\n{'Layer':>5} {'PR orig':>8} {'PR resid':>9} {'Dom %':>7} {'90% comp':>9} {'KV-Shared':>10}")
print('-' * 50)
for l in layers:
    r = results[l]
    shared = 'YES' if l >= kv_shared_start else ''
    print(f"{l:5d} {r['pr_original']:8.1f} {r['pr_residual']:9.1f} {r['dominant_var_share']:7.1%} "
          f"{r['n90_residual']:9d} {shared:>10}")

print(f"\nQwen3-0.6B zum Vergleich (aus Sim 2b):")
print(f"  Layer 7:  PR 1.3 -> 120.5 (dom: 100.0%)")
print(f"  Layer 14: PR 1.5 -> 127.4 (dom: 100.0%)")
print(f"  Layer 21: PR 3.9 -> 127.2 (dom: 99.3%)")

## Sim 3: Parallele Verarbeitung mit Mean-Separation

Gleicher Test wie Sim 3 auf Qwen: Subspaces parallel verarbeiten, Output vergleichen.

In [ ]:
# Batch-Aktivierungen fuer Hook-basierte Verarbeitung
best_layer = max(results, key=lambda l: results[l]['pr_residual'])
print(f"Bester Layer (hoechste Residual-PR): {best_layer}")

acts_batch, tokens = helpers.extract_activations_batched(
    model, tokenizer, all_prompts, [best_layer, best_layer + 1]
)

# Ground truth: normaler Output von Layer best_layer+1
ground_truth = acts_batch[best_layer + 1]

# Parallel forward mit Mean-Separation
k_values = [2, 3, 4, 5]
results_pda = {"k": [], "cos_sim": [], "mse": []}

for k in k_values:
    print(f"\nk={k}: Parallele Verarbeitung...")
    outputs = helpers.parallel_forward_mean_separated(
        model, tokenizer, acts_batch[best_layer], best_layer, k, tokens
    )
    
    if outputs:
        merged = torch.stack(outputs).mean(dim=0)
        
        cos = F.cosine_similarity(
            merged.reshape(-1, d_model),
            ground_truth.reshape(-1, d_model), dim=1
        ).mean().item()
        
        mse = torch.mean((merged - ground_truth)**2).item()
        
        results_pda['k'].append(k)
        results_pda['cos_sim'].append(cos)
        results_pda['mse'].append(mse)
        print(f"  CosSim: {cos:.4f}, MSE: {mse:.4f}")
    else:
        print(f"  FEHLER: Keine Outputs")

In [ ]:
# Sim 3: Output-Qualitaet (Token Accuracy)
print(f"\nOutput-Qualitaet Test (Layer {best_layer}, k=2)...")

with torch.no_grad():
    original_logits = model(tokens.to(model.device)).logits.cpu()

# PDA logits: inject merged activations at best_layer+1
outputs_k2 = helpers.parallel_forward_mean_separated(
    model, tokenizer, acts_batch[best_layer], best_layer, 2, tokens
)
merged_k2 = torch.stack(outputs_k2).mean(dim=0)

# Inject and get logits
captured_logits = {}
def inject_and_capture(merged_act):
    def inject_hook(module, input, output):
        if isinstance(output, tuple):
            return (merged_act.to(model.device),) + output[1:]
        return merged_act.to(model.device)
    return inject_hook

h = model.model.layers[best_layer + 1].register_forward_hook(inject_and_capture(merged_k2))
with torch.no_grad():
    pda_logits = model(tokens.to(model.device)).logits.cpu()
h.remove()

# Compare
top1_orig = original_logits.argmax(dim=-1)
top1_pda = pda_logits.argmax(dim=-1)
accuracy = (top1_orig == top1_pda).float().mean().item()

kl = F.kl_div(
    F.log_softmax(pda_logits, dim=-1),
    F.softmax(original_logits, dim=-1),
    reduction='batchmean'
).item()

print(f"  Top-1 Token Accuracy: {accuracy:.4f}")
print(f"  KL Divergence:        {kl:.4f}")
print(f"\n  Qwen3-0.6B zum Vergleich (Sim 3, L24, k=2):")
print(f"  Top-1: 0.675, KL: 0.408")

In [ ]:
# Visualisierung
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar([str(k) for k in results_pda['k']], results_pda['cos_sim'], color='steelblue')
ax1.set_title(f'CosSim vs k (Layer {best_layer})')
ax1.set_xlabel('k subspaces')
ax1.set_ylabel('Cosine Similarity')
ax1.set_ylim(0, 1)
ax1.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='0.9')
ax1.grid(True, axis='y')
ax1.legend()

ax2.bar([str(k) for k in results_pda['k']], results_pda['mse'], color='coral')
ax2.set_title('MSE vs k')
ax2.set_xlabel('k subspaces')
ax2.set_ylabel('MSE')
ax2.grid(True, axis='y')

plt.suptitle(f'Gemma-4 Sim 3: PDA Parallel Forward ({MODEL_ID})', fontsize=14)
plt.tight_layout()
plt.show()

## Gemma-4 Spezifisch: KV-Shared vs Non-Shared Layer

Vergleich der Subspace-Struktur in Layern die KV teilen vs. eigenes KV haben.

In [ ]:
if kv_shared_start < n_layers:
    # Teste einen Layer VOR und NACH der KV-Shared-Grenze
    test_layers = [
        max(0, kv_shared_start - 2),  # 2 vor der Grenze
        kv_shared_start - 1,           # direkt vor
        kv_shared_start,               # erster shared
        min(n_layers - 2, kv_shared_start + 2),  # 2 nach
    ]
    test_layers = sorted(set(test_layers))
    
    print(f"KV-Shared Grenze: Layer {kv_shared_start}")
    print(f"Teste Layer: {test_layers}")
    
    acts_kv = helpers.extract_activations(model, tokenizer, all_prompts, test_layers)
    
    print(f"\n{'Layer':>5} {'PR orig':>8} {'PR resid':>9} {'Dom %':>7} {'KV-Shared':>10}")
    print('-' * 42)
    for l in test_layers:
        r = helpers.analyze_dominant_direction(acts_kv[l])
        shared = 'SHARED' if l >= kv_shared_start else ''
        print(f"{l:5d} {r['pr_original']:8.1f} {r['pr_residual']:9.1f} {r['dominant_var_share']:7.1%} {shared:>10}")
else:
    print("Kein KV-Sharing in diesem Modell konfiguriert.")

## Zusammenfassung

In [ ]:
print('=' * 70)
print('GEMMA-4 SIM 2b + 3: ERGEBNIS-ZUSAMMENFASSUNG')
print('=' * 70)

print(f'\nModell: {MODEL_ID}')
print(f'Layer: {n_layers}, d_model: {d_model}')
print(f'PLE: {info.get("has_ple", False)}, KV-Shared ab: {kv_shared_start}')

print(f'\n--- Sim 2b: Dominant Direction ---')
for l in sorted(results.keys()):
    r = results[l]
    shared = ' [KV-SHARED]' if l >= kv_shared_start else ''
    print(f'  L{l}: PR {r["pr_original"]:.1f} -> {r["pr_residual"]:.1f} '
          f'(dom: {r["dominant_var_share"]:.1%}){shared}')

print(f'\n--- Sim 3: Parallel Forward (Layer {best_layer}) ---')
for i, k in enumerate(results_pda['k']):
    print(f'  k={k}: CosSim={results_pda["cos_sim"][i]:.4f}')

print(f'\n--- Output-Qualitaet (k=2) ---')
print(f'  Top-1 Accuracy: {accuracy:.4f}')
print(f'  KL Divergence:  {kl:.4f}')

print(f'\n--- Vergleich mit Qwen3 ---')
print(f'  Qwen3-0.6B Sim 2b: PR 1.3->120 (dom 100%), CosSim 0.93 (k=2)')
print(f'  Qwen3-0.6B Sim 3:  Top-1 0.675, KL 0.408 (L24, k=2)')

best_cos = max(results_pda['cos_sim']) if results_pda['cos_sim'] else 0
print(f'\n{"="*70}')
if accuracy > 0.8:
    print('ENTSCHEIDUNG: Gemma-4 zeigt BESSERE PDA-Eignung als Qwen3.')
    print('  PLE und/oder Shared-KV veraendern die Subspace-Struktur positiv.')
elif accuracy > 0.675:  # besser als Qwen3
    print('ENTSCHEIDUNG: Gemma-4 zeigt ETWAS BESSERE PDA-Eignung als Qwen3.')
    print('  Verbesserung messbar, aber nicht dramatisch.')
else:
    print('ENTSCHEIDUNG: Gemma-4 zeigt AEHNLICHE PDA-Eignung wie Qwen3.')
    print('  Architektur-Unterschiede aendern das Grundproblem nicht.')
print('=' * 70)